In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sympy as smp
from scipy.integrate import quad


In [ ]:


# Plot style: white background with a clearly visible grid (cosmetic only)
sns.set_style("whitegrid")
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["grid.color"] = "black"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 1
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["axes.titlecolor"] = "black"



# Applied Root-Finding Problems

Three applied engineering problems, each reduced to a single-variable root-finding problem and
solved with an open method (Newton's method or fixed-point iteration).

### A. Channel depth from Manning's equation

From fluid mechanics, Manning's semi-empirical formula for the design of rectangular channels is
written as:
$$
Q = \frac{1}{n} \frac{\left(B H\right)^{\frac{5}{3}} S^{\frac{1}{2}}}{\left(B + 2H\right)^{\frac{2}{3}}}
$$

where Q is the flow rate, n is the roughness coefficient between the liquid and the channel walls,
B is the channel width, H is the channel depth, and S is the slope (tangent of the angle the channel
forms with the horizontal). A channel must drain 50 m3/s of water, over a surface with roughness
0.045, in a space limited to 35 m of width. Determine the required channel height if the slope must
not exceed 0.1°.

Substituting the known values and solving analytically for H reduces the problem to the following
root-finding equation:

$$
0.0189675 (35 + 2H)^{\frac{2}{3}} - H^{\frac{5}{3}} = 0
$$


Solved with Newton's method:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function Q(H)
def Q(H):
    return 0.0189675 * (35 + 2 * H)**(2/3) - H**(5/3)

# Approximate derivative of Q(H)
def dQ(H, dH=1e-6):
    return (Q(H + dH) - Q(H)) / dH

# Newton's method
def newton_method():
    error = 1
    tol = 1e-6
    attempts = 0
    max_attempts = 2
    iteration = 0
    max_iter = 100

    # Propose an initial value H0
    valid = False
    while not valid and error > tol and attempts < max_attempts:
        try:
            H0 = float(input("Propose an initial value H0: "))
            if abs(dQ(H0)) > 0:
                valid = True
            else:
                print("The derivative of Q(H0) is zero or too small. Try another value.")
            if Q(H0) == 0:
                error = 0
        except ValueError:
            print("Invalid input. Enter a numeric value.")
        attempts += 1

    if not valid:
        print("No suitable initial value was found.")
        return None

    # Newton's method iteration
    H_prev = H0
    iterations = []
    errors = []

    while valid and error > tol and iteration < max_iter:
        H_new = H_prev - Q(H_prev) / dQ(H_prev)
        error = abs((H_new - H_prev) / H_new)
        iterations.append(iteration)
        errors.append(error)

        iteration += 1
        H_prev = H_new

    # Results
    print("\nSolution: the required channel height for a slope not exceeding 0.1 deg is:", H_prev)
    print("Function evaluated at the solution Q(H):", Q(H_prev))
    print("Error reached:", error)
    print("Number of iterations:", iteration)

    # Plot of the function and the root
    H_vals = np.linspace(0, 20, 1000)
    Q_vals = Q(H_vals)

    plt.figure(figsize=(12, 8))
    plt.plot(H_vals, Q_vals, label='Q(H)', color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.scatter(H_prev, 0, color='red', s=100, label=f'Root: {H_prev:.6f}')
    plt.xlabel('H')
    plt.ylabel('Q(H)')
    plt.legend()
    plt.grid(True)
    plt.title("Approximate Root via Newton's Method")
    plt.savefig("manning_equation_root.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Iterations for different target errors
    test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]
    test_iteration_counts = []

    for tol in test_tolerances:
        H_prev = H0
        error = 1
        iteration = 0
        while error > tol and iteration < max_iter:
            H_new = H_prev - Q(H_prev) / dQ(H_prev)
            error = abs((H_new - H_prev) / H_new)
            H_prev = H_new
            iteration += 1
        test_iteration_counts.append(iteration)

    # Iterations vs. target error plot
    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, test_iteration_counts, marker='o', color='green', label='Iterations vs. Target Error')
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.grid(True)
    plt.legend()
    plt.title("Number of Iterations vs. Target Error (Newton's Method)")
    plt.savefig("manning_equation_iterations_vs_error.png", dpi=150, bbox_inches="tight")
    plt.show()

# Run Newton's method
newton_method()


### B. Equilibrium angle of a loaded bar

A bar supports a 200 lb weight and is pinned at end A. It is subject to a 100 lb-ft couple moment.
The spring has an unstretched length of 2 ft and stiffness k = 50 lb/ft. Determine the angle theta
required for equilibrium.

The force/moment analysis reduces to the following expression in terms of theta:

$$
1800 \sin(\theta) \cos(\theta) - 100 - 600 \cos(\theta) = 0
$$

Theta is renamed to x purely for notational convenience in the code. Newton's method is used here,
since fixed-point iteration (tried first, with several candidate g(x)) did not converge for this
equation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function f(x)
def f(x):
    return 1800*np.sin(x)*np.cos(x) - 100 - 600*np.cos(x)

# Approximate derivative of f(x)
def df(x, dx=1e-6):
    return (f(x + dx) - f(x)) / dx

# Newton's method
def newton_method():
    error = 1
    tol = 1e-6
    attempts = 0
    max_attempts = 2
    iteration = 0
    max_iter = 100

    # Propose an initial value x0
    valid = False
    while not valid and error > tol and attempts < max_attempts:
        try:
            x0 = float(input("Propose an x0: "))
            if abs(df(x0)) > 0:
                valid = True
            else:
                print("The derivative of f(x0) is zero or too small. Try another value.")
            if f(x0) == 0:
                error = 0
        except ValueError:
            print("Invalid input. Enter a numeric value.")
        attempts += 1

    if not valid:
        print("No suitable initial value was found.")
        return None

    # Newton's method iteration
    x_prev = x0
    iterations = []
    errors = []

    while valid and error > tol and iteration < max_iter:
        x_new = x_prev - f(x_prev) / df(x_prev)
        error = abs((x_new - x_prev) / x_new)
        iterations.append(iteration)
        errors.append(error)

        iteration += 1
        x_prev = x_new

    # Results
    print("\nSolution, the equilibrium angle theta is:", x_prev)
    print("Function evaluated at the solution:", f(x_prev))
    print("Error reached:", error)
    print("Number of iterations:", iteration)

    # Plot of the function and the root
    x_vals = np.linspace(-1, 2, 1000)
    y_vals = f(x_vals)

    plt.figure(figsize=(12, 8))
    plt.plot(x_vals, y_vals, label='f(x)', color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.scatter(x_prev, 0, color='red', s=100, label=f'Root: {x_prev:.6f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.title("Approximate Root via Newton's Method")
    plt.show()


    # Iterations for different target errors
    test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]
    test_iteration_counts = []

    for tol in test_tolerances:
        x_prev = x0
        error = 1
        iteration = 0
        while error > tol and iteration < max_iter:
            x_new = x_prev - f(x_prev) / df(x_prev)
            error = abs((x_new - x_prev) / x_new)
            x_prev = x_new
            iteration += 1
        test_iteration_counts.append(iteration)

    # Iterations vs. target error plot
    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, test_iteration_counts, marker='o', color='green', label='Iterations vs. Target Error')
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.grid(True)
    plt.legend()
    plt.title("Number of Iterations vs. Target Error (Newton's Method)")
    plt.show()

# Run Newton's method
newton_method()


### C. Launch angle for a projectile with a fixed range

The trajectory of a projectile in projectile motion follows the general equation:
$$
y = \tan(\theta_0) x - \frac{g x^2}{2 v_0^2 \cos^2(\theta_0)} + y_0
$$
Find the appropriate launch angle theta_0 if the initial speed is 20 m/s and the horizontal distance
traveled is 40 m. The object is launched from an initial height of 1.8 m and ends its motion touching
the ground.

As before, theta_0 is renamed to x for notational convenience, and substituting the known values and
solving reduces the problem to the following root-finding equation:
$$
40 \cos^2(\theta_0) \tan(\theta_0) + 1.8 \cos^2(\theta_0) - 19.62 = 0
$$

Attempted first with fixed-point iteration:

In [ ]:

# Function f(x) and iteration function g(x)
def f(x):
    return 40*(np.cos(x)**2) * np.tan(x) + (1.8)*np.cos(x)**2 - 19.62

def g(x):
    return np.arctan((19.62 - 1.8*np.cos(x)**2) / (40*np.cos(x)**2))  # g(x) obtained by isolating tan(x) in f(x) = 0

# Approximate derivative of g(x)
def dg(x, dx=1e-6):
    return (g(x + dx) - g(x)) / dx

# Fixed-point iteration
def fixed_point_method():
    error = 1
    tol = 1e-6
    attempts = 0
    max_attempts = 2
    iteration = 0
    max_iter = 100

    # Propose an initial value x0
    valid = False
    while not valid and error > tol and attempts < max_attempts:
        try:
            x0 = float(input("Propose an x0: "))
            if abs(dg(x0)) < 1:
                valid = True
            else:
                print("The derivative of g(x0) is >= 1. Try another value.")
            if f(x0) == 0:
                error = 0
        except ValueError:
            print("Invalid input. Enter a numeric value.")
        attempts += 1

    if not valid:
        print("No suitable initial value was found.")
        return None

    # Fixed-point iteration loop
    x_prev = x0
    iterations = []
    errors = []

    while valid and error > tol and iteration < max_iter:
        x_new = g(x_prev)
        error = abs((x_new - x_prev) / x_new)
        iterations.append(iteration)
        errors.append(error)

        if abs(dg(x_new)) >= 1:
            valid = False

        iteration += 1
        x_prev = x_new

    # Results
    print("\nSolution, the appropriate launch angle theta0 for the given conditions is:", x_prev)
    print("Function evaluated at the solution:", f(x_prev))
    print("Error reached:", error)
    print("Number of iterations:", iteration)

    # Plot of the function and the root
    x_vals = np.linspace(-1, 2, 1000)
    y_vals = f(x_vals)

    plt.figure(figsize=(12, 8))
    plt.plot(x_vals, y_vals, label='f(x)', color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.scatter(x_prev, 0, color='red', s=100, label=f'Root: {x_prev:.6f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.title('Approximate Root via Fixed-Point Iteration')
    plt.show()


    # Iterations for different target errors
    test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]
    test_iteration_counts = []

    for tol in test_tolerances:
        x_prev = x0
        error = 1
        iteration = 0
        while error > tol and iteration < max_iter:
            x_new = g(x_prev)
            error = abs((x_new - x_prev) / x_new)
            x_prev = x_new
            iteration += 1
        test_iteration_counts.append(iteration)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, test_iteration_counts, marker='o', color='green', label='Iterations vs. Target Error')
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.grid(True)
    plt.legend()
    plt.title('Number of Iterations vs. Target Error (Fixed-Point Iteration)')
    plt.show()
    
fixed_point_method()



Fixed-point iteration converges to the root for this g(x); `scipy.optimize.brentq` below is used as an independent cross-check of the result:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Function to plot (same f(x) as above)
def f(x):
    return 40*(np.cos(x)**2) * np.tan(x) + (1.8)*np.cos(x)**2 - 19.62
# Plot range
x_vals = np.linspace(0.5, 0.65, 1000)
y_vals = f(x_vals)

# Find a root, if one exists in the given range
try:
    root = brentq(f, 0.5, 0.65)
    print(f"Root found: x = {root:.6f}, f(x) = {f(root):.6e}")
except ValueError:
    root = None
    print("No root found in the given range.")

# Plot the function
plt.figure(figsize=(12, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')

# Mark the root if found
if root is not None:
    plt.scatter(root, 0, color='red', s=100, label=f'Root: x = {root:.6f}')

plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Function Plot and Its Intersection with the x-Axis')
plt.legend()
plt.grid(True)
plt.show()
